In [1]:
# ============================================================
# GENETIC ALGORITHM-BASED FEATURE SELECTION
# FOR BREAST CANCER CLASSIFICATION
# ============================================================

# 1. IMPORT LIBRARIES
import numpy as np
import pandas as pd

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)


# ============================================================
# 2. LOAD BREAST CANCER DATASET
# ============================================================

data = load_breast_cancer()

X = pd.DataFrame(
    data.data,
    columns=data.feature_names
)

y = pd.Series(data.target)

print("========================================")
print("BREAST CANCER DATASET")
print("========================================")
print("Number of samples  :", X.shape[0])
print("Number of features :", X.shape[1])
print("Target classes     :", data.target_names)
print()


# ============================================================
# 3. TRAIN-TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples :", X_train.shape[0])
print("Testing samples  :", X_test.shape[0])
print()


# ============================================================
# 4. FEATURE SCALING
# ============================================================

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


# ============================================================
# 5. CHROMOSOME CREATION
# ============================================================

def create_chromosome(n_features):

    # 0 = feature not selected
    # 1 = feature selected

    chromosome = np.random.randint(
        0,
        2,
        n_features
    )

    # At least one feature must be selected
    if np.sum(chromosome) == 0:
        chromosome[
            np.random.randint(0, n_features)
        ] = 1

    return chromosome


# ============================================================
# 6. CREATE INITIAL POPULATION
# ============================================================

def create_population(population_size, n_features):

    population = []

    for i in range(population_size):

        chromosome = create_chromosome(
            n_features
        )

        population.append(chromosome)

    return np.array(population)


# ============================================================
# 7. FITNESS FUNCTION
# ============================================================

def fitness_function(chromosome, X, y):

    # Convert chromosome to Boolean mask
    selected = chromosome.astype(bool)

    # If no feature is selected
    if np.sum(selected) == 0:
        return 0

    # Select only chosen features
    X_selected = X[:, selected]

    # Machine Learning model
    model = RandomForestClassifier(
        n_estimators=100,
        random_state=42
    )

    # Train model
    model.fit(
        X_selected,
        y
    )

    # Prediction
    prediction = model.predict(
        X_selected
    )

    # Accuracy
    accuracy = accuracy_score(
        y,
        prediction
    )

    # Number of selected features
    selected_features = np.sum(selected)

    # Feature penalty
    feature_ratio = (
        selected_features /
        len(chromosome)
    )

    # Fitness
    fitness = (
        accuracy -
        0.01 * feature_ratio
    )

    return fitness


# ============================================================
# 8. TOURNAMENT SELECTION
# ============================================================

def tournament_selection(
    population,
    fitnesses,
    tournament_size=3
):

    parents = []

    for i in range(2):

        # Randomly select chromosomes
        indices = np.random.choice(
            len(population),
            tournament_size,
            replace=False
        )

        # Find chromosome with best fitness
        winner = indices[
            np.argmax(
                fitnesses[indices]
            )
        ]

        parents.append(
            population[winner]
        )

    return parents


# ============================================================
# 9. CROSSOVER
# ============================================================

def crossover(parent1, parent2):

    # Random crossover point
    point = np.random.randint(
        1,
        len(parent1)
    )

    # Create child 1
    child1 = np.concatenate([
        parent1[:point],
        parent2[point:]
    ])

    # Create child 2
    child2 = np.concatenate([
        parent2[:point],
        parent1[point:]
    ])

    return child1, child2


# ============================================================
# 10. MUTATION
# ============================================================

def mutation(
    chromosome,
    mutation_rate=0.1
):

    chromosome = chromosome.copy()

    for i in range(len(chromosome)):

        # Mutation condition
        if np.random.random() < mutation_rate:

            # Flip bit
            if chromosome[i] == 0:
                chromosome[i] = 1
            else:
                chromosome[i] = 0

    # Make sure at least one feature is selected
    if np.sum(chromosome) == 0:

        chromosome[
            np.random.randint(
                0,
                len(chromosome)
            )
        ] = 1

    return chromosome


# ============================================================
# 11. GENETIC ALGORITHM
# ============================================================

def genetic_algorithm(
    X,
    y,
    population_size=20,
    generations=20,
    mutation_rate=0.1
):

    n_features = X.shape[1]

    # Initial population
    population = create_population(
        population_size,
        n_features
    )

    best_chromosome = None
    best_fitness = -np.inf

    print("========================================")
    print("GENETIC ALGORITHM STARTED")
    print("========================================")

    # Generations
    for generation in range(generations):

        # Calculate fitness for every chromosome
        fitnesses = np.array([

            fitness_function(
                chromosome,
                X,
                y
            )

            for chromosome in population

        ])

        # Find best chromosome
        best_index = np.argmax(
            fitnesses
        )

        current_best_fitness = (
            fitnesses[best_index]
        )

        # Update global best
        if current_best_fitness > best_fitness:

            best_fitness = (
                current_best_fitness
            )

            best_chromosome = (
                population[
                    best_index
                ].copy()
            )

        # Display generation result
        print(
            f"Generation {generation + 1:02d} | "
            f"Fitness = {best_fitness:.4f} | "
            f"Features = "
            f"{np.sum(best_chromosome)}"
        )

        # New population
        new_population = []

        while len(new_population) < population_size:

            # Selection
            parent1, parent2 = (
                tournament_selection(
                    population,
                    fitnesses
                )
            )

            # Crossover
            child1, child2 = crossover(
                parent1,
                parent2
            )

            # Mutation
            child1 = mutation(
                child1,
                mutation_rate
            )

            child2 = mutation(
                child2,
                mutation_rate
            )

            new_population.append(
                child1
            )

            new_population.append(
                child2
            )

        # Keep population size fixed
        population = np.array(
            new_population[
                :population_size
            ]
        )

    print()
    print("GENETIC ALGORITHM FINISHED")

    return best_chromosome, best_fitness


# ============================================================
# 12. RUN GENETIC ALGORITHM
# ============================================================

best_chromosome, best_fitness = genetic_algorithm(
    X_train_scaled,
    y_train,
    population_size=20,
    generations=20,
    mutation_rate=0.1
)


# ============================================================
# 13. GET SELECTED FEATURES
# ============================================================

selected_indices = np.where(
    best_chromosome == 1
)[0]

selected_features = X.columns[
    selected_indices
]

print()
print("========================================")
print("SELECTED FEATURES")
print("========================================")

for i, feature in enumerate(
    selected_features,
    start=1
):

    print(
        f"{i}. {feature}"
    )

print()
print(
    "Original number of features :",
    X.shape[1]
)

print(
    "Selected number of features :",
    len(selected_features)
)

print(
    "Best fitness                :",
    round(best_fitness, 4)
)


# ============================================================
# 14. CREATE DATA USING SELECTED FEATURES
# ============================================================

X_train_selected = (
    X_train_scaled[
        :,
        selected_indices
    ]
)

X_test_selected = (
    X_test_scaled[
        :,
        selected_indices
    ]
)


# ============================================================
# 15. RANDOM FOREST - ALL FEATURES
# ============================================================

rf_all = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

rf_all.fit(
    X_train_scaled,
    y_train
)

prediction_all = rf_all.predict(
    X_test_scaled
)

probability_all = (
    rf_all.predict_proba(
        X_test_scaled
    )[:, 1]
)


# ============================================================
# 16. RANDOM FOREST - GA SELECTED FEATURES
# ============================================================

rf_ga = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

rf_ga.fit(
    X_train_selected,
    y_train
)

prediction_ga = rf_ga.predict(
    X_test_selected
)

probability_ga = (
    rf_ga.predict_proba(
        X_test_selected
    )[:, 1]
)


# ============================================================
# 17. EVALUATION FUNCTION
# ============================================================

def evaluate_model(
    y_true,
    prediction,
    probability
):

    accuracy = accuracy_score(
        y_true,
        prediction
    )

    precision = precision_score(
        y_true,
        prediction
    )

    recall = recall_score(
        y_true,
        prediction
    )

    f1 = f1_score(
        y_true,
        prediction
    )

    auc = roc_auc_score(
        y_true,
        probability
    )

    return [
        accuracy,
        precision,
        recall,
        f1,
        auc
    ]


# ============================================================
# 18. CALCULATE RESULTS
# ============================================================

results_all = evaluate_model(
    y_test,
    prediction_all,
    probability_all
)

results_ga = evaluate_model(
    y_test,
    prediction_ga,
    probability_ga
)


# ============================================================
# 19. RESULTS TABLE
# ============================================================

results = pd.DataFrame({

    "Model": [
        "Random Forest - All Features",
        "Random Forest - GA Features"
    ],

    "Features": [
        X.shape[1],
        len(selected_features)
    ],

    "Accuracy": [
        results_all[0],
        results_ga[0]
    ],

    "Precision": [
        results_all[1],
        results_ga[1]
    ],

    "Recall": [
        results_all[2],
        results_ga[2]
    ],

    "F1 Score": [
        results_all[3],
        results_ga[3]
    ],

    "ROC-AUC": [
        results_all[4],
        results_ga[4]
    ]

})


# ============================================================
# 20. DISPLAY RESULTS
# ============================================================

print()
print("========================================")
print("FINAL RESULTS")
print("========================================")

print(
    results.to_string(
        index=False
    )
)


# ============================================================
# 21. CONFUSION MATRIX
# ============================================================

print()
print("========================================")
print("CONFUSION MATRIX - GA MODEL")
print("========================================")

cm = confusion_matrix(
    y_test,
    prediction_ga
)

print(cm)


# ============================================================
# 22. CLASSIFICATION REPORT
# ============================================================

print()
print("========================================")
print("CLASSIFICATION REPORT - GA MODEL")
print("========================================")

print(
    classification_report(
        y_test,
        prediction_ga,
        target_names=data.target_names
    )
)

BREAST CANCER DATASET
Number of samples  : 569
Number of features : 30
Target classes     : ['malignant' 'benign']

Training samples : 455
Testing samples  : 114

GENETIC ALGORITHM STARTED
Generation 01 | Fitness = 0.9967 | Features = 10
Generation 02 | Fitness = 0.9967 | Features = 10
Generation 03 | Fitness = 0.9970 | Features = 9
Generation 04 | Fitness = 0.9980 | Features = 6
Generation 05 | Fitness = 0.9980 | Features = 6
Generation 06 | Fitness = 0.9980 | Features = 6
Generation 07 | Fitness = 0.9980 | Features = 6
Generation 08 | Fitness = 0.9980 | Features = 6
Generation 09 | Fitness = 0.9987 | Features = 4
Generation 10 | Fitness = 0.9987 | Features = 4
Generation 11 | Fitness = 0.9987 | Features = 4
Generation 12 | Fitness = 0.9987 | Features = 4
Generation 13 | Fitness = 0.9987 | Features = 4
Generation 14 | Fitness = 0.9987 | Features = 4
Generation 15 | Fitness = 0.9987 | Features = 4
Generation 16 | Fitness = 0.9987 | Features = 4
Generation 17 | Fitness = 0.9987 | Featur